# Key-Value Attention for Thai Karaoke Character-level Machine Translation (Many-to-Many, encoder-decoder)

In this homework, you will create an MT model with attention mechnism that coverts names of Thai 2019 MP candidates from Thai script to Roman(Latin) script. E.g. นิยม-->niyom

The use of Pytorch Lightning is optional but recommended. You can use Pytorch if you prefer.

In [2]:
!pip install lightning wandb
!wget https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: topten-watthana (topten-watthana-chulalongkorn-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
%matplotlib inline
import matplotlib as mpl
mpl.font_manager.fontManager.addfont('thsarabunnew-webfont.ttf') # 3.2+
mpl.rc('font', family='TH Sarabun New')
import torch
# import torchtext
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import lightning as L
import numpy as np

import random

## Load Dataset
We have generated a toy dataset using names of Thai MP candidates in 2019 Thai General Election from elect.in.th's github(https://github.com/codeforthailand/dataset-election-62-candidates) and tltk (https://pypi.org/project/tltk/) library to convert them into Roman script.

```
ไกรสีห์ kraisi
พัชรี phatri
ธีระ thira
วุฒิกร wutthikon
ไสว sawai
สัมภาษณ์  samphat
วศิน wasin
ทินวัฒน์ thinwat
ศักดินัย sakdinai
สุรศักดิ์ surasak
```


In [5]:
!wget https://raw.githubusercontent.com/ekapolc/nlp_2019/master/HW8/mp_name_th_en.csv

--2025-02-25 16:07:27--  https://raw.githubusercontent.com/ekapolc/nlp_2019/master/HW8/mp_name_th_en.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 324399 (317K) [text/plain]
Saving to: ‘mp_name_th_en.csv’

mp_name_th_en.csv   100%[===================>] 316.80K  --.-KB/s    in 0.03s   

2025-02-25 16:07:27 (11.8 MB/s) - ‘mp_name_th_en.csv’ saved [324399/324399]



In [6]:
import csv

with open('mp_name_th_en.csv') as csvfile:
    readCSV = csv.reader(csvfile, delimiter=',')
    name_th = []
    name_en = []
    for row in readCSV:
        temp_th = row[0]
        temp_en = row[1]

        name_th.append(temp_th)
        name_en.append(temp_en)

In [7]:
for th, en in zip(name_th[:10],name_en[:10]):
    print(th,en)

ไกรสีห์ kraisi
พัชรี phatri
ธีระ thira
วุฒิกร wutthikon
ไสว sawai
สัมภาษณ์  samphat
วศิน wasin
ทินวัฒน์ thinwat
ศักดินัย sakdinai
สุรศักดิ์ surasak


## TODO1: Preprocess dataset
* You will need 2 vocabularies (1 for input and another for output)
* DON'T FORGET TO INCLUDE special token for padding (for both input and output)
* DON'T FORGET TO INCLUDE special token for the end of word symbol (output)

In [8]:
#Preprocessing
input_chars = list(set(''.join(name_th)))
output_chars = list(set(''.join(name_en)))
data_size, vocab_size = len(name_th), len(input_chars)+1
output_vocab_size = len(output_chars)+2#+2 for special end of sentence token/PADDING
print('There are %d lines and %d unique characters in your input data.' % (data_size, vocab_size))
maxlen = len( max(name_th, key=len)) #max input length
maxlen_out = len( max(name_en, key=len)) #max input length

There are 10887 lines and 65 unique characters in your input data.


In [9]:
print("Max input length:", maxlen)
print("Max output length:", maxlen_out)

Max input length: 20
Max output length: 19


In [10]:
from torch.utils.data import Dataset, DataLoader

In [11]:
input_vocab = {char: i+1 for i, char in enumerate(sorted(input_chars))}
# For output, we reserve index 0 for padding and use (len(output_char_to_idx)+1) as EOS.
output_vocab = {char: i+1 for i, char in enumerate(sorted(output_chars))}
EOS_token = len(output_vocab) + 1

class NameDataset(Dataset):
  def __init__(self, X, y, input_mapping, output_mapping, maxlen, maxlen_out):
        self.X = X
        self.y = y
        self.input_mapping = input_mapping
        self.output_mapping = output_mapping
        self.maxlen = maxlen
        self.maxlen_out = maxlen_out  # without EOS token padding

  def __getitem__(self, idx):
        name_th = self.X[idx]
        name_en = self.y[idx]

        # Convert input (Thai) into indices and pad.
        x_indices = [self.input_mapping[c] for c in name_th]
        x_indices = x_indices + [0] * (self.maxlen - len(x_indices))

        # Convert output (English) into indices, append EOS token, and pad.
        y_indices = [self.output_mapping[c] for c in name_en]
        y_indices.append(EOS_token)  # add end-of-word token
        # Adjust max output length (+1 for EOS)
        y_indices = y_indices + [0] * ((self.maxlen_out + 1) - len(y_indices))

        return torch.tensor(x_indices, dtype=torch.long), torch.tensor(y_indices, dtype=torch.long)

  def __len__(self):
        return len(self.X)

In [46]:
print(input_vocab)


{' ': 1, 'ก': 2, 'ข': 3, 'ค': 4, 'ฆ': 5, 'ง': 6, 'จ': 7, 'ฉ': 8, 'ช': 9, 'ซ': 10, 'ฌ': 11, 'ญ': 12, 'ฎ': 13, 'ฏ': 14, 'ฐ': 15, 'ฑ': 16, 'ฒ': 17, 'ณ': 18, 'ด': 19, 'ต': 20, 'ถ': 21, 'ท': 22, 'ธ': 23, 'น': 24, 'บ': 25, 'ป': 26, 'ผ': 27, 'ฝ': 28, 'พ': 29, 'ฟ': 30, 'ภ': 31, 'ม': 32, 'ย': 33, 'ร': 34, 'ล': 35, 'ว': 36, 'ศ': 37, 'ษ': 38, 'ส': 39, 'ห': 40, 'ฬ': 41, 'อ': 42, 'ฮ': 43, 'ะ': 44, 'ั': 45, 'า': 46, 'ำ': 47, 'ิ': 48, 'ี': 49, 'ึ': 50, 'ื': 51, 'ุ': 52, 'ู': 53, 'เ': 54, 'แ': 55, 'โ': 56, 'ใ': 57, 'ไ': 58, '็': 59, '่': 60, '้': 61, '๊': 62, '๋': 63, '์': 64}


In [12]:
class NameDataModule(L.LightningDataModule):

  def __init__(self, train_data, y, batch_size, input_mapping, output_mapping, maxlen, maxlen_out, num_workers=0):
        super().__init__()
        self.train_data = train_data
        self.y = y
        self.batch_size = batch_size
        self.input_mapping = input_mapping
        self.output_mapping = output_mapping
        self.maxlen = maxlen
        self.maxlen_out = maxlen_out
        self.num_workers = num_workers


  def setup(self, stage: str = None):
        self.train_dataset = NameDataset(
            X=self.train_data,
            y=self.y,
            input_mapping=self.input_mapping,
            output_mapping=self.output_mapping,
            maxlen=self.maxlen,
            maxlen_out=self.maxlen_out
        )
  def collate_fn(self, batch):
        xs, ys = zip(*batch)
        xs = torch.stack(xs)
        ys = torch.stack(ys)
        return {'x': xs, 'y': ys}

  def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            collate_fn=self.collate_fn,
            shuffle=True
        )

# Attention Mechanism


## TODO 2: Code your own (key-value) attention mechnism
* PLEASE READ: you DO NOT have to follow all the details in (Daniluk, et al. 2017). You just need to create a key-value attention mechanism where the "key" part of the mechanism is used for attention score calculation, and the "value" part of the mechanism is used to encode information to create a context vector.  
* fill code for one_step_attention function



In [13]:
def one_step_attention(h, s_prev, W1, W2, V):

    #Split into Key-Value
    #do concat with s_prev.
    #hint: you will need to use s_prev.repeat(...) somehow so that it has the same dimension as the key
    #hint2: s_prev.unsqueeze() could also be useful


    #Attention function###
    # use layer(s) from your model to calculate attention_scores and then softmax
    # calculate a context vector

    key, value = torch.split(h, h.size(2)//2, dim=2)  # key, value: (batch_size, seq_len, hidden_size)

    s_prev_expanded = s_prev.unsqueeze(1).repeat(1, key.size(1), 1)  # (batch_size, seq_len, hidden_size)

    energy = torch.tanh(W1(key) + W2(s_prev_expanded))  # (batch_size, seq_len, attn_dim)

    scores = V(energy).squeeze(2)  # (batch_size, seq_len)

    attn_weights = F.softmax(scores, dim=1)  # (batch_size, seq_len)

    context = torch.bmm(attn_weights.unsqueeze(1), value).squeeze(1)  # (batch_size, hidden_size)

    return context,attn_weights

# Translation Model

## TODO3: Create and train your encoder/decoder model here

In [32]:
class AttentionModel(L.LightningModule):
    def __init__(self, input_vocab_size, output_vocab_size, max_output_length=20):
        super().__init__()
        self.n_h = 128  # hidden dimension สำหรับ encoder
        self.n_s = 128  # hidden dimension สำหรับ decoder
        self.max_output_length = max_output_length
        self.learning_rate = 0.001

        # Encoder
        self.encoder_embedding = nn.Embedding(input_vocab_size, self.n_h)
        self.encoder = nn.LSTM(self.n_h, self.n_h, batch_first=True, bidirectional=True)
        # Decoder
        self.decoder_lstm_cell = nn.LSTMCell(self.n_s, self.n_s)
        self.output_layer = nn.Linear(self.n_s, output_vocab_size)

        # Attention layers
        self.attn_W1 = nn.Linear(self.n_h, self.n_s)
        self.attn_W2 = nn.Linear(self.n_s, self.n_s)
        self.attn_V = nn.Linear(self.n_s, 1)

        self.criterion = nn.CrossEntropyLoss()

    def forward(self, src, return_attention=False):
        embedded = self.encoder_embedding(src)  # (batch_size, seq_len, n_h)
        h, _ = self.encoder(embedded)  # (batch_size, seq_len, 2*n_h)

        decoder_s = torch.randn(src.shape[0], self.n_s).to(self.device)
        decoder_c = torch.randn(src.shape[0], self.n_s).to(self.device)

        # ใช้ self.max_output_length และ self.output_layer.out_features ที่ถูกต้อง
        prediction = torch.zeros((src.shape[0], self.max_output_length, self.output_layer.out_features)).to(self.device)
        attention_scores = []
        for t in range(self.max_output_length):
            context, attn_score = one_step_attention(h, decoder_s, self.attn_W1, self.attn_W2, self.attn_V)
            attention_scores.append(attn_score)
            decoder_s, decoder_c = self.decoder_lstm_cell(context, (decoder_s, decoder_c))
            out = self.output_layer(decoder_s)
            prediction[:, t] = out

        if return_attention:
            return prediction, attention_scores
        else:
            return prediction

    def training_step(self, batch, batch_idx):
        try:
              src = batch['x']
              target = batch['y']
              print("target min:", target.min().item(), "target max:", target.max().item())
              print("unique target values:", torch.unique(target))
              prediction = self(src)
              prediction = prediction.reshape(-1, self.output_layer.out_features)
              target = target.reshape(-1)
              loss = self.criterion(prediction, target)
              self.log("train_loss", loss)
              return loss
        except Exception as e:
             print("Error in training_step:", e)
             raise e


    def predict_step(self, batch, batch_idx, dataloader_idx=0):
            # รับ input จาก batch เป็น dictionary ที่มี key 'x'
            src = batch['x']
            with torch.no_grad():
               prediction = self(src, return_attention=False)
            return prediction


    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.learning_rate)


In [33]:
model = AttentionModel(input_vocab_size=len(input_vocab)+1,  # +1 สำหรับ padding
                         output_vocab_size=len(output_vocab)+2,  #
                         max_output_length=20)

In [34]:
batch_size = 32
data_module = NameDataModule(
    train_data=name_th,
    y=name_en,
    batch_size=batch_size,
    num_workers=0,
    input_mapping=input_vocab,
    output_mapping=output_vocab,
    maxlen=maxlen,
    maxlen_out=maxlen_out
)


In [35]:
from lightning import Trainer
from lightning.pytorch.loggers import WandbLogger
wandb_logger = WandbLogger(project="hw3.1_attention")

In [36]:
trainer = L.Trainer(
    max_epochs=100,
    logger=wandb_logger
)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [19]:
# %env CUDA_LAUNCH_BLOCKING=1


In [37]:
print(trainer.max_epochs)


100


In [38]:
# model.to('cpu')
# หากใช้ DataModule ให้แน่ใจว่าข้อมูลถูกย้ายไป CPU ด้วย
trainer.fit(model, data_module)


/usr/local/lib/python3.11/dist-packages/lightning/pytorch/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory ./hw3.1_attention/5mq17i8q/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name              | Type             | Params | Mode 
---------------------------------------------------------------
0 | encoder_embedding | Embedding        | 8.3 K  | train
1 | encoder           | LSTM             | 264 K  | train
2 | decoder_lstm_cell | LSTMCell         | 132 K  | train
3 | output_layer      | Linear           | 3.1 K  | train
4 | attn_W1           | Linear           | 16.5 K | train
5 |

Training: |          | 0/? [00:00<?, ?it/s]

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
unique target values: tensor([ 0,  2,  3,  4,  5,  6,  8,  9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20,
        21, 22, 23], device='cuda:0')
target min: 0 target max: 23
unique target values: tensor([ 0,  2,  4,  5,  6,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
        21, 22, 23], device='cuda:0')
target min: 0 target max: 23
unique target values: tensor([ 0,  2,  3,  4,  6,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
        21, 22, 23], device='cuda:0')
target min: 0 target max: 23
unique target values: tensor([ 0,  2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
        20, 21, 22, 23], device='cuda:0')
target min: 0 target max: 23
unique target values: tensor([ 0,  2,  3,  4,  5,  6,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
        20, 21, 22, 23], device='cuda:0')
target min: 0 target max: 23
unique target values: tensor([ 0,  2,  3,  4,  5,  6,  8,  9, 10, 11, 13, 14, 15, 16, 17, 

INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


target min: 0 target max: 23
unique target values: tensor([ 0,  2,  5,  6,  8,  9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23],
       device='cuda:0')


# Test Your Model

## TODO4: Test your model on 5 examples of your choice including your name!

Example Output:
```
prayutthatha</s></s>aa</s></s>a</s>
somchai</s></s></s></s>a</s></s>a</s></s></s></s></s>
thanathon</s></s></s></s></s></s></s></s></s></s></s>
newin</s>i</s></s></s></s></s></s></s></s></s></s></s></s></s>
suthep</s>he</s></s></s></s></s></s></s></s></s></s></s>
prawit</s></s></s></s></s></s></s></s></s></s></s></s></s></s>
chatchachatti</s></s>i</s></s></s></s>
```

<font color='blue'>Paste your model predictions in MyCourseVille</font>

In [43]:
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# ตัวอย่างชื่อที่ต้องการทดสอบ (เลือก 5-7 ตัวอย่าง รวมชื่อของคุณด้วย)
EXAMPLES = ['ประยุทธ', 'สมชาย', 'ธนาธร', 'เนวิน', 'สุเทพ', 'ประวิตร์', 'ชัชชาติ']

# Dataset สำหรับ prediction
class PredictDataset(Dataset):
    def __init__(self, examples, input_mapping, maxlen):
        self.examples = examples
        self.input_mapping = input_mapping
        self.maxlen = maxlen

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        text = self.examples[idx]
        # แปลงแต่ละตัวอักษรเป็น index; หากไม่พบใช้ 0 (PAD)
        indices = [self.input_mapping.get(char, 0) for char in text]
        # เติม padding ให้ครบ maxlen
        if len(indices) < self.maxlen:
            indices += [0] * (self.maxlen - len(indices))
        else:
            indices = indices[:self.maxlen]
        return torch.tensor(indices, dtype=torch.long)

def collate_fn_predict(batch):
    return {"x": torch.stack(batch)}

# สร้าง DataLoader สำหรับ prediction
predict_dataset = PredictDataset(EXAMPLES, input_vocab, maxlen)
predict_loader = DataLoader(predict_dataset, batch_size=1, shuffle=False)

# ฟังก์ชันสำหรับ decode ผลลัพธ์ (แปลง index กลับเป็น token)
def decode_prediction(pred_tensor, rev_mapping, eos_token):
    """
    pred_tensor: tensor ขนาด (max_output_length, output_vocab_size)
    rev_mapping: reverse mapping จาก index เป็น token สำหรับ output_vocab
    eos_token: EOS token index
    """
    # คำนวณ argmax สำหรับแต่ละ timestep
    pred_indices = torch.argmax(F.softmax(pred_tensor, dim=-1), dim=-1)
    tokens = []
    for idx in pred_indices:
        idx = idx.item()
        if idx == 0:
            # PAD token ให้แสดงเป็น </s>
            tokens.append("</s>")
        elif idx == eos_token:
            tokens.append("</s>")
        else:
            tokens.append(rev_mapping.get(idx, "?"))
    return "".join(tokens)

# สร้าง reverse mapping สำหรับ output_vocab
rev_output_vocab = {v: k for k, v in output_vocab.items()}

In [47]:
for i in range(len(predict_dataset)):
    x = predict_dataset[i]
    print(x, x.dtype)


tensor([26, 34, 44, 33, 52, 22, 23,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0]) torch.int64
tensor([39, 32,  9, 46, 33,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0]) torch.int64
tensor([23, 24, 46, 23, 34,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0]) torch.int64
tensor([54, 24, 36, 48, 24,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0]) torch.int64
tensor([39, 52, 54, 22, 29,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0]) torch.int64
tensor([26, 34, 44, 36, 48, 20, 34, 64,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0]) torch.int64
tensor([ 9, 45,  9,  9, 46, 20, 48,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0]) torch.int64


In [44]:
model.eval()

AttentionModel(
  (encoder_embedding): Embedding(65, 128)
  (encoder): LSTM(128, 128, batch_first=True, bidirectional=True)
  (decoder_lstm_cell): LSTMCell(128, 128)
  (output_layer): Linear(in_features=128, out_features=24, bias=True)
  (attn_W1): Linear(in_features=128, out_features=128, bias=True)
  (attn_W2): Linear(in_features=128, out_features=128, bias=True)
  (attn_V): Linear(in_features=128, out_features=1, bias=True)
  (criterion): CrossEntropyLoss()
)

In [45]:
predictions = trainer.predict(model, predict_loader)

print("Translation Results:")
for ex, pred in zip(EXAMPLES, predictions):
    # pred มี shape (batch_size=1, max_output_length, output_vocab_size)
    # ดึง tensor ใน batch แรก (เพราะ batch_size=1)
    pred_tensor = pred[0]
    decoded = decode_prediction(pred_tensor, rev_output_vocab, EOS_token)
    print(f"Input: {ex}")
    print(f"Prediction: {decoded}")

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

TypeError: new(): invalid data type 'str'

In [24]:
output = trainer.predict(model, predict_loader)

NameError: name 'predict_loader' is not defined

## TODO 5: Show your visualization of attention scores on one of your example

<font color='blue'>Paste your visualization image in MyCourseVille</font>

In [ ]:
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
prediction, attention_scores = zip(*output)

In [ ]:
ax = sns.heatmap(attn_viz, linewidth=0.5)
ax.set_yticklabels(output_text,rotation=30)
ax.set_xticklabels(xlabels,rotation=60)
plt.show()